In [0]:
"""
sim_invoice_delta.py
====================
Simulation script: generates synthetic Accounting / Invoice data and writes
it into Databricks Delta tables under  smart_odoo.silver

Tables covered (11 total):
    1.  silver.account_account              ← 120 new rows per run
    2.  silver.account_analytic_account     ← 30  new rows per run
    3.  silver.account_analytic_line        ← 1 line per move line (~600/run)
    4.  silver.account_bank_statement       ← 20  new rows per run
    5.  silver.account_bank_statement_line  ← 2–4 lines per statement
    6.  silver.account_move                 ← 200 new rows per run
    7.  silver.account_move_line            ← 3   lines per move (600/run)
    8.  silver.account_journal              ← fixed 5 journals; seeded once
    9.  silver.account_tax                  ← fixed 3 taxes;    seeded once
   10.  silver.account_payment              ← 1 per paid/partial move
   11.  silver.account_payment_term         ← fixed 4 terms;    seeded once

════════════════════════════════════════════════════════════
SIMULATION RULES
════════════════════════════════════════════════════════════

[GENERAL]
- Catalog / Schema   : smart_odoo.silver
- Source tag         : dwh_source_db = "python_sim"
- Date range         : 2025-01-01  →  2026-04-22
- All IDs            : MAX(existing_id) + 1  →  append-safe, zero duplicates
- company_id         : always 1  ("Smart Odoo LLC")
- partner_id         : 1 – 50
- product_id         : 1 – 100
- TAX_RATE           : 14 % (Egypt VAT standard)

[FIXED REFERENCE TABLES]  — seeded once, skipped on re-runs
  account_journal (5)     : Sales | Purchase | Bank | Cash | General
  account_tax (3)         : VAT 14 % | VAT 5 % | Exempt
  account_payment_term (4): Immediate | Net 30 | Net 60 | 2/10 Net 30

[account_account]  — 120 new rows per run
- account_type: asset_current | asset_receivable | asset_cash | asset_fixed |
                liability_current | liability_payable | equity |
                income | expense | expense_direct_cost
- code_store   : prefix (e.g. "101") + random 3-digit suffix
- reconcile    : True only for asset_receivable / liability_payable
- non_trade    : True only for equity accounts
- currency_id  : None (70 %) | 1-3 (30 %)

[account_analytic_account]  — 30 new rows per run
- plan_id      : 1–3  (Cost Center | Project | Department)
- code         : ANA-NNNNN
- partner_id   : nullable 40 %

[account_move]  (invoices/bills)  — 200 new rows per run
- move_type    : out_invoice (50 %) | in_invoice (50 %)
- state        : always "posted"
- payment_state: not_paid (50 %) | partial (30 %) | paid (20 %)
- amount tiers :
    small  (50 %): 300   – 3,000
    medium (35 %): 3,000 – 25,000
    large  (15 %): 25,000 – 150,000
- amount_residual  : 0 if paid | 20–70 % of total if partial | total if not_paid
- invoice_date_due : date + 15 | 30 | 45 days
- delivery_date    : nullable (40 %)
- reversed_entry_id: nullable (~5 % credit notes)
- sign convention  : out_invoice amounts positive, in_invoice amounts negative (signed fields)

[account_move_line]  — 3 lines per move (always balanced: debit = credit)
  out_invoice lines:
    1. Receivable account  → debit = total,   credit = 0
    2. Income account      → debit = 0,       credit = base
    3. Tax account         → debit = 0,       credit = tax_amount
  in_invoice lines:
    1. Expense account     → debit = base,    credit = 0
    2. Tax account         → debit = tax,     credit = 0
    3. Payable account     → debit = 0,       credit = total
- quantity     : 1–20
- price_unit   : base / quantity
- discount     : 0 (70 %) | 5 | 10 | 15 %
- reconciled   : True when payment_state = "paid"

[account_payment]  — 1 per paid or partial move
- payment_type : outbound (in_invoice) | inbound (out_invoice)
- partner_type : supplier (in_invoice) | customer (out_invoice)
- state        : posted
- payment_method_id : 1=Manual | 2=Bank Transfer | 3=Check (weighted)
- is_reconciled: True when payment_state = "paid"
- is_matched   : True when payment_state = "paid"
- amount       : total if paid | partial amount if partial
- amount_company_currency_signed: negative for outbound

[account_bank_statement]  — 20 per run
- journal_id   : Bank (3) or Cash (4) journals only
- balance_end  : balance_start + net of lines
- is_complete  : True ~60 %

[account_bank_statement_line]  — 2–4 per statement
- transaction_type : credit | debit (weighted 60/40)
- is_reconciled    : True ~55 %
- amount_residual  : 0 if reconciled | full amount if not

[account_analytic_line]  — 1 per account_move_line (product lines only)
- amount       : mirrors move line price_subtotal (negative for expenses)
- unit_amount  : mirrors quantity
- category     : invoice | expense

════════════════════════════════════════════════════════════
"""

import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType,
    BooleanType, TimestampType
)

# ─────────────────────────────────────────────────────────────
# SPARK + CATALOG
# ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_invoice_delta").getOrCreate()

CATALOG    = "smart_odoo"
SCHEMA     = "silver"
SOURCE_DB  = "python_sim"
COMPANY_ID = 1
COMPANY    = "Smart Odoo LLC"
TAX_RATE   = 0.14

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────
START_DATE = datetime(2025, 1, 1)
END_DATE   = datetime(2026, 4, 22)

def rand_date() -> datetime:
    return START_DATE + timedelta(
        days=random.randint(0, (END_DATE - START_DATE).days)
    )

def maybe(val, prob_none: float = 0.3):
    return None if random.random() < prob_none else val

now = datetime.now()

PARTNERS  = {i: f"Partner_{i:03d}" for i in range(1, 51)}
PRODUCTS  = {i: f"Product_{i:03d}" for i in range(1, 101)}
UOMS      = {1: "Units", 2: "kg", 3: "liters", 4: "pcs", 5: "boxes"}
CURRENCIES= {1: ("EGP", 1.0), 2: ("USD", 48.5), 3: ("EUR", 52.0)}

def existing_names(table: str, col: str = "name") -> set:
    try:
        return {r[col] for r in spark.sql(
            f"SELECT {col} FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()}
    except Exception:
        return set()

def max_id(table: str, col: str) -> int:
    try:
        return spark.sql(
            f"SELECT COALESCE(MAX({col}), 0) FROM {CATALOG}.{SCHEMA}.{table}"
        ).collect()[0][0]
    except Exception:
        return 0

def append_delta(df, table: str):
    (df.write.format("delta").mode("append")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    print(f"  ✅  {table}  (+{df.count()} rows)")


# ═══════════════════════════════════════════════════════════════
# 8. ACCOUNT_JOURNAL  (5 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
JOURNAL_SEED = [
    (1, "SAL",  "Sales Journal",    "sale",     1),
    (2, "PUR",  "Purchase Journal", "purchase", 2),
    (3, "BNK",  "Bank Journal",     "bank",     3),
    (4, "CSH",  "Cash Journal",     "cash",     4),
    (5, "MISC", "General Journal",  "general",  5),
]
JOURNALS     = {row[0]: (row[1], row[2], row[3]) for row in JOURNAL_SEED}
SALE_JIDS    = [1]
PURCHASE_JIDS= [2]
BANK_JIDS    = [3, 4]

existing_journal_names = existing_names("account_journal")
j_rows = []
for jid, code, jname, jtype, seq in JOURNAL_SEED:
    if jname in existing_journal_names:
        continue
    cur_name, _ = CURRENCIES[1]
    j_rows.append(Row(
        account_journal_id        = jid,
        company_id                = COMPANY_ID,
        company_name              = COMPANY,
        currency_id               = 1,
        currency_name             = "EGP",
        default_account_id        = None,
        default_account_name      = None,
        sequence                  = seq,
        code                      = code,
        type                      = jtype,
        name                      = jname,
        bank_statements_source    = "undefined" if jtype not in ("bank","cash") else "file_import",
        invoice_reference_type    = "sequence",
        invoice_reference_model   = "odoo",
        active                    = True,
        is_self_billing           = False,
        restrict_mode_hash_table  = False,
        refund_sequence           = True,
        payment_sequence          = True,
        show_on_dashboard         = jtype in ("bank", "cash"),
        created_at                = now,
        updated_at                = now,
        dwh_loaded_at             = now,
        dwh_source_db             = SOURCE_DB,
    ))

j_schema = StructType([
    StructField("account_journal_id",        IntegerType(),   True),
    StructField("company_id",                IntegerType(),   True),
    StructField("company_name",              StringType(),    True),
    StructField("currency_id",               IntegerType(),   True),
    StructField("currency_name",             StringType(),    True),
    StructField("default_account_id",        IntegerType(),   True),
    StructField("default_account_name",      StringType(),    True),
    StructField("sequence",                  IntegerType(),   True),
    StructField("code",                      StringType(),    True),
    StructField("type",                      StringType(),    True),
    StructField("name",                      StringType(),    True),
    StructField("bank_statements_source",    StringType(),    True),
    StructField("invoice_reference_type",    StringType(),    True),
    StructField("invoice_reference_model",   StringType(),    True),
    StructField("active",                    BooleanType(),   True),
    StructField("is_self_billing",           BooleanType(),   True),
    StructField("restrict_mode_hash_table",  BooleanType(),   True),
    StructField("refund_sequence",           BooleanType(),   True),
    StructField("payment_sequence",          BooleanType(),   True),
    StructField("show_on_dashboard",         BooleanType(),   True),
    StructField("created_at",                TimestampType(), True),
    StructField("updated_at",                TimestampType(), True),
    StructField("dwh_loaded_at",             TimestampType(), True),
    StructField("dwh_source_db",             StringType(),    True),
])

if j_rows:
    append_delta(spark.createDataFrame(j_rows, schema=j_schema), "account_journal")
else:
    print("  ⏭   account_journal  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 9. ACCOUNT_TAX  (3 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
TAX_SEED = [
    (1, "VAT 14%",  14.0, "sale",     1, "T1", "S"),
    (2, "VAT 5%",    5.0, "purchase", 1, "T2", "S"),
    (3, "Exempt",    0.0, "sale",     1, "T3", "E"),
]
TAXES = {row[0]: (row[1], row[2]) for row in TAX_SEED}

existing_tax_names = existing_names("account_tax")
tax_rows = []
for tid, tname, amount, use, tgrp, eta, ubl in TAX_SEED:
    if tname in existing_tax_names:
        continue
    tax_rows.append(Row(
        account_tax_id           = tid,
        company_id               = COMPANY_ID,
        company_name             = COMPANY,
        tax_group_id             = tgrp,
        tax_group_name           = f"Tax Group {tgrp}",
        country_id               = 1,
        country_name             = "Egypt",
        sequence                 = tid,
        type_tax_use             = use,
        tax_scope                = None,
        amount_type              = "percent",
        tax_exigibility          = "on_invoice",
        name                     = tname,
        description              = tname,
        invoice_label            = tname,
        ubl_cii_tax_category_code= ubl,
        l10n_eg_eta_code         = eta,
        amount                   = amount,
        is_domestic              = True,
        active                   = True,
        include_base_amount      = False,
        is_base_affected         = True,
        analytic                 = False,
        created_at               = now,
        updated_at               = now,
        dwh_loaded_at            = now,
        dwh_source_db            = SOURCE_DB,
    ))

tax_schema = StructType([
    StructField("account_tax_id",            IntegerType(),   True),
    StructField("company_id",                IntegerType(),   True),
    StructField("company_name",              StringType(),    True),
    StructField("tax_group_id",              IntegerType(),   True),
    StructField("tax_group_name",            StringType(),    True),
    StructField("country_id",                IntegerType(),   True),
    StructField("country_name",              StringType(),    True),
    StructField("sequence",                  IntegerType(),   True),
    StructField("type_tax_use",              StringType(),    True),
    StructField("tax_scope",                 StringType(),    True),
    StructField("amount_type",               StringType(),    True),
    StructField("tax_exigibility",           StringType(),    True),
    StructField("name",                      StringType(),    True),
    StructField("description",               StringType(),    True),
    StructField("invoice_label",             StringType(),    True),
    StructField("ubl_cii_tax_category_code", StringType(),    True),
    StructField("l10n_eg_eta_code",          StringType(),    True),
    StructField("amount",                    DoubleType(),    True),
    StructField("is_domestic",               BooleanType(),   True),
    StructField("active",                    BooleanType(),   True),
    StructField("include_base_amount",       BooleanType(),   True),
    StructField("is_base_affected",          BooleanType(),   True),
    StructField("analytic",                  BooleanType(),   True),
    StructField("created_at",                TimestampType(), True),
    StructField("updated_at",                TimestampType(), True),
    StructField("dwh_loaded_at",             TimestampType(), True),
    StructField("dwh_source_db",             StringType(),    True),
])

if tax_rows:
    append_delta(spark.createDataFrame(tax_rows, schema=tax_schema), "account_tax")
else:
    print("  ⏭   account_tax  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 11. ACCOUNT_PAYMENT_TERM  (4 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
PT_SEED = [
    (1, "Immediate Payment", 1,  0,    None,  0.0,  False, True),
    (2, "Net 30",            2,  30,   None,  0.0,  False, True),
    (3, "Net 60",            3,  60,   None,  0.0,  False, True),
    (4, "2/10 Net 30",       4,  30,   10,    2.0,  True,  True),
]
# (id, name, seq, days, discount_days, discount_pct, early_disc, display)
PAYMENT_TERMS = {row[0]: row[1] for row in PT_SEED}

existing_pt_names = existing_names("account_payment_term")
pt_rows = []
for ptid, ptname, seq, days, ddays, dpct, edisc, display in PT_SEED:
    if ptname in existing_pt_names:
        continue
    pt_rows.append(Row(
        account_payment_term_id          = ptid,
        company_id                       = COMPANY_ID,
        company_name                     = COMPANY,
        sequence                         = seq,
        discount_days                    = ddays,
        early_pay_discount_computation   = "mixed" if edisc else None,
        name                             = ptname,
        note                             = None,
        discount_percentage              = dpct,
        active                           = True,
        display_on_invoice               = display,
        early_discount                   = edisc,
        created_at                       = now,
        updated_at                       = now,
        dwh_loaded_at                    = now,
        dwh_source_db                    = SOURCE_DB,
    ))

pt_schema = StructType([
    StructField("account_payment_term_id",         IntegerType(),   True),
    StructField("company_id",                      IntegerType(),   True),
    StructField("company_name",                    StringType(),    True),
    StructField("sequence",                        IntegerType(),   True),
    StructField("discount_days",                   IntegerType(),   True),
    StructField("early_pay_discount_computation",  StringType(),    True),
    StructField("name",                            StringType(),    True),
    StructField("note",                            StringType(),    True),
    StructField("discount_percentage",             DoubleType(),    True),
    StructField("active",                          BooleanType(),   True),
    StructField("display_on_invoice",              BooleanType(),   True),
    StructField("early_discount",                  BooleanType(),   True),
    StructField("created_at",                      TimestampType(), True),
    StructField("updated_at",                      TimestampType(), True),
    StructField("dwh_loaded_at",                   TimestampType(), True),
    StructField("dwh_source_db",                   StringType(),    True),
])

if pt_rows:
    append_delta(spark.createDataFrame(pt_rows, schema=pt_schema), "account_payment_term")
else:
    print("  ⏭   account_payment_term  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# RESOLVE ALL APPEND IDs
# ═══════════════════════════════════════════════════════════════
acc_id_start    = max_id("account_account",             "account_id")           + 1
ana_id_start    = max_id("account_analytic_account",    "analytic_account_id")  + 1
anal_id_start   = max_id("account_analytic_line",       "analytic_line_id")     + 1
bs_id_start     = max_id("account_bank_statement",      "bank_statement_id")    + 1
bsl_id_start    = max_id("account_bank_statement_line", "bank_statement_line_id")+ 1
move_id_start   = max_id("account_move",                "account_move_id")      + 1
line_id_start   = max_id("account_move_line",           "account_move_line_id") + 1
pay_id_start    = max_id("account_payment",             "account_payment_id")   + 1

print(f"account_id start             : {acc_id_start}")
print(f"analytic_account_id start    : {ana_id_start}")
print(f"analytic_line_id start       : {anal_id_start}")
print(f"bank_statement_id start      : {bs_id_start}")
print(f"bank_statement_line_id start : {bsl_id_start}")
print(f"account_move_id start        : {move_id_start}")
print(f"account_move_line_id start   : {line_id_start}")
print(f"account_payment_id start     : {pay_id_start}")


# ═══════════════════════════════════════════════════════════════
# 1. ACCOUNT_ACCOUNT  (120 per run)
# ═══════════════════════════════════════════════════════════════
ACC_TYPES = [
    "asset_current", "asset_receivable", "asset_cash", "asset_fixed",
    "liability_current", "liability_payable", "equity",
    "income", "expense", "expense_direct_cost"
]
NAMES_MAP = {
    "asset_current":        ["Current Assets", "Short Term Assets", "Liquid Assets"],
    "asset_receivable":     ["Accounts Receivable", "Customer Receivables"],
    "asset_cash":           ["Cash", "Bank"],
    "asset_fixed":          ["Fixed Assets", "Equipment"],
    "liability_current":    ["Current Liabilities"],
    "liability_payable":    ["Accounts Payable"],
    "equity":               ["Capital", "Owner Equity"],
    "income":               ["Sales Revenue", "Operating Income"],
    "expense":              ["Operating Expenses"],
    "expense_direct_cost":  ["COGS"],
}
CODE_PREFIX = {
    "asset_current": "101", "asset_receivable": "121", "asset_cash": "1014",
    "asset_fixed": "151", "liability_current": "201", "liability_payable": "211",
    "equity": "301", "income": "400", "expense": "600", "expense_direct_cost": "500"
}

acc_rows = []
acc_id   = acc_id_start
acc_pool: dict[str, list[int]] = {t: [] for t in ACC_TYPES}  # used later for move lines

for _ in range(120):
    atype     = random.choice(ACC_TYPES)
    aname     = random.choice(NAMES_MAP[atype])
    code      = f"{CODE_PREFIX[atype]}{random.randint(100, 999)}"
    cur_id    = None if random.random() < 0.7 else random.randint(1, 3)
    cur_name  = CURRENCIES[cur_id][0] if cur_id else None
    reconcile = atype in ("asset_receivable", "liability_payable")
    non_trade = atype == "equity"
    cdate     = rand_date()
    wdate     = cdate + timedelta(days=random.randint(0, 5))

    acc_rows.append(Row(
        account_id    = acc_id,
        currency_id   = cur_id,
        currency_name = cur_name,
        account_type  = atype,
        name          = aname,
        code_store    = code,
        note          = f"{aname} account",
        active        = True,
        reconcile     = reconcile,
        non_trade     = non_trade,
        created_at    = cdate,
        updated_at    = wdate,
        dwh_loaded_at = now,
        dwh_source_db = SOURCE_DB,
    ))
    acc_pool[atype].append(acc_id)
    acc_id += 1

acc_schema = StructType([
    StructField("account_id",    IntegerType(),   True),
    StructField("currency_id",   IntegerType(),   True),
    StructField("currency_name", StringType(),    True),
    StructField("account_type",  StringType(),    True),
    StructField("name",          StringType(),    True),
    StructField("code_store",    StringType(),    True),
    StructField("note",          StringType(),    True),
    StructField("active",        BooleanType(),   True),
    StructField("reconcile",     BooleanType(),   True),
    StructField("non_trade",     BooleanType(),   True),
    StructField("created_at",    TimestampType(), True),
    StructField("updated_at",    TimestampType(), True),
    StructField("dwh_loaded_at", TimestampType(), True),
    StructField("dwh_source_db", StringType(),    True),
])

# Also pull existing accounts from Delta for FK pool
try:
    for r in spark.sql(
        f"SELECT account_id, account_type FROM {CATALOG}.{SCHEMA}.account_account"
    ).collect():
        if r.account_type in acc_pool:
            acc_pool[r.account_type].append(r.account_id)
except Exception:
    pass

def pick_acc(atype: str) -> int:
    pool = acc_pool.get(atype, [])
    return random.choice(pool) if pool else acc_id_start


# ═══════════════════════════════════════════════════════════════
# 2. ACCOUNT_ANALYTIC_ACCOUNT  (30 per run)
# ═══════════════════════════════════════════════════════════════
PLANS = {1: "Cost Center", 2: "Project", 3: "Department"}
ana_rows = []
ana_id   = ana_id_start
ana_pool = []

for _ in range(30):
    plan_id = random.choice(list(PLANS.keys()))
    pid     = maybe(random.randint(1, 50), 0.4)
    ana_rows.append(Row(
        analytic_account_id = ana_id,
        plan_id             = plan_id,
        plan_name           = PLANS[plan_id],
        company_id          = COMPANY_ID,
        company_name        = COMPANY,
        partner_id          = pid,
        partner_name        = PARTNERS[pid] if pid else None,
        code                = f"ANA-{ana_id:05d}",
        name                = f"{PLANS[plan_id]} {ana_id}",
        active              = True,
        created_at          = now,
        updated_at          = now,
        dwh_loaded_at       = now,
        dwh_source_db       = SOURCE_DB,
    ))
    ana_pool.append(ana_id)
    ana_id += 1

ana_schema = StructType([
    StructField("analytic_account_id", IntegerType(),   True),
    StructField("plan_id",             IntegerType(),   True),
    StructField("plan_name",           StringType(),    True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("partner_id",          IntegerType(),   True),
    StructField("partner_name",        StringType(),    True),
    StructField("code",                StringType(),    True),
    StructField("name",                StringType(),    True),
    StructField("active",              BooleanType(),   True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

# Also load existing analytic accounts
try:
    for r in spark.sql(
        f"SELECT analytic_account_id FROM {CATALOG}.{SCHEMA}.account_analytic_account"
    ).collect():
        ana_pool.append(r.analytic_account_id)
except Exception:
    pass


# ═══════════════════════════════════════════════════════════════
# 4. ACCOUNT_BANK_STATEMENT  (20 per run)
# ═══════════════════════════════════════════════════════════════
bs_rows  = []
bsl_rows = []
bs_id    = bs_id_start
bsl_id   = bsl_id_start

for i in range(20):
    jid      = random.choice(BANK_JIDS)
    jcode, jname, _ = JOURNALS[jid]
    cur_id   = 1
    bdate    = rand_date()
    bal_s    = round(random.uniform(10_000, 500_000), 2)
    is_comp  = random.random() < 0.6

    # lines first to compute balance_end
    num_lines = random.randint(2, 4)
    lines_data = []
    net = 0.0
    for seq in range(1, num_lines + 1):
        t_type   = random.choice(["credit"] * 60 + ["debit"] * 40)
        amount   = round(random.uniform(500, 20_000), 2) * (1 if t_type == "credit" else -1)
        rec      = random.random() < 0.55
        residual = 0.0 if rec else abs(amount)
        pid      = random.randint(1, 50)
        lines_data.append((seq, t_type, amount, rec, residual, pid))
        net += amount

    bal_e     = round(bal_s + net, 2)
    bal_e_real= round(bal_e * random.uniform(0.98, 1.02), 2)

    bs_rows.append(Row(
        bank_statement_id = bs_id,
        company_id        = COMPANY_ID,
        company_name      = COMPANY,
        currency_id       = cur_id,
        currency_name     = "EGP",
        journal_id        = jid,
        journal_name      = jname,
        name              = f"BNK/{bdate.year}/{bs_id:05d}",
        reference         = maybe(f"REF-{random.randint(1000,9999)}", 0.4),
        balance_start     = bal_s,
        balance_end       = bal_e,
        balance_end_real  = bal_e_real,
        is_complete       = is_comp,
        date              = bdate,
        created_at        = bdate,
        updated_at        = bdate,
        dwh_loaded_at     = now,
        dwh_source_db     = SOURCE_DB,
    ))

    for seq, t_type, amount, rec, residual, pid in lines_data:
        fcur_id = maybe(random.randint(2, 3), 0.7)
        bsl_rows.append(Row(
            bank_statement_line_id = bsl_id,
            statement_id           = bs_id,
            move_id                = None,
            move_name              = None,
            journal_id             = jid,
            journal_name           = jname,
            company_id             = COMPANY_ID,
            company_name           = COMPANY,
            partner_id             = pid,
            partner_name           = PARTNERS[pid],
            currency_id            = cur_id,
            currency_name          = "EGP",
            foreign_currency_id    = fcur_id,
            foreign_currency_name  = CURRENCIES[fcur_id][0] if fcur_id else None,
            sequence               = seq,
            account_number         = f"EG{random.randint(10**18, 10**19-1)}",
            transaction_type       = t_type,
            payment_ref            = f"PAY-{random.randint(10000,99999)}",
            amount                 = amount,
            amount_currency        = round(amount * random.uniform(0.9, 1.1), 2),
            amount_residual        = residual,
            is_reconciled          = rec,
            created_at             = bdate,
            updated_at             = bdate,
            dwh_loaded_at          = now,
            dwh_source_db          = SOURCE_DB,
        ))
        bsl_id += 1

    bs_id += 1

bs_schema = StructType([
    StructField("bank_statement_id", IntegerType(),   True),
    StructField("company_id",        IntegerType(),   True),
    StructField("company_name",      StringType(),    True),
    StructField("currency_id",       IntegerType(),   True),
    StructField("currency_name",     StringType(),    True),
    StructField("journal_id",        IntegerType(),   True),
    StructField("journal_name",      StringType(),    True),
    StructField("name",              StringType(),    True),
    StructField("reference",         StringType(),    True),
    StructField("balance_start",     DoubleType(),    True),
    StructField("balance_end",       DoubleType(),    True),
    StructField("balance_end_real",  DoubleType(),    True),
    StructField("is_complete",       BooleanType(),   True),
    StructField("date",              TimestampType(), True),
    StructField("created_at",        TimestampType(), True),
    StructField("updated_at",        TimestampType(), True),
    StructField("dwh_loaded_at",     TimestampType(), True),
    StructField("dwh_source_db",     StringType(),    True),
])

bsl_schema = StructType([
    StructField("bank_statement_line_id", IntegerType(),   True),
    StructField("statement_id",           IntegerType(),   True),
    StructField("move_id",                IntegerType(),   True),
    StructField("move_name",              StringType(),    True),
    StructField("journal_id",             IntegerType(),   True),
    StructField("journal_name",           StringType(),    True),
    StructField("company_id",             IntegerType(),   True),
    StructField("company_name",           StringType(),    True),
    StructField("partner_id",             IntegerType(),   True),
    StructField("partner_name",           StringType(),    True),
    StructField("currency_id",            IntegerType(),   True),
    StructField("currency_name",          StringType(),    True),
    StructField("foreign_currency_id",    IntegerType(),   True),
    StructField("foreign_currency_name",  StringType(),    True),
    StructField("sequence",               IntegerType(),   True),
    StructField("account_number",         StringType(),    True),
    StructField("transaction_type",       StringType(),    True),
    StructField("payment_ref",            StringType(),    True),
    StructField("amount",                 DoubleType(),    True),
    StructField("amount_currency",        DoubleType(),    True),
    StructField("amount_residual",        DoubleType(),    True),
    StructField("is_reconciled",          BooleanType(),   True),
    StructField("created_at",             TimestampType(), True),
    StructField("updated_at",             TimestampType(), True),
    StructField("dwh_loaded_at",          TimestampType(), True),
    StructField("dwh_source_db",          StringType(),    True),
])


# ═══════════════════════════════════════════════════════════════
# 6 + 7 + 10.  ACCOUNT_MOVE → MOVE_LINES → PAYMENTS  (200 moves)
# ═══════════════════════════════════════════════════════════════
PAYMENT_METHODS = {1: "Manual", 2: "Bank Transfer", 3: "Check"}
PM_POOL = [1]*60 + [2]*30 + [3]*10

move_rows  = []
ml_rows    = []
pay_rows   = []
anal_rows  = []

move_id  = move_id_start
line_id  = line_id_start
pay_id   = pay_id_start
anal_id  = anal_id_start

for _ in range(200):

    move_type = random.choice(["out_invoice", "in_invoice"])
    is_out    = move_type == "out_invoice"

    jid       = random.choice(SALE_JIDS if is_out else PURCHASE_JIDS)
    jcode, jname, _ = JOURNALS[jid]

    pid       = random.randint(1, 50)
    user_id   = random.randint(1, 10)
    cur_id    = random.choice([1, 1, 1, 2, 3])   # mostly EGP
    cur_name, cur_rate = CURRENCIES[cur_id]
    pt_id     = random.choice(list(PAYMENT_TERMS.keys()))

    date      = rand_date()
    due_days  = random.choice([15, 30, 45])
    date_due  = date + timedelta(days=due_days)
    del_date  = maybe(date + timedelta(days=random.randint(1, 10)), 0.4)

    # amount tier
    tier = random.random()
    if tier < 0.50:
        base = round(random.uniform(300, 3_000), 2)
    elif tier < 0.85:
        base = round(random.uniform(3_000, 25_000), 2)
    else:
        base = round(random.uniform(25_000, 150_000), 2)

    tax_amount = round(base * TAX_RATE, 2)
    total      = round(base + tax_amount, 2)

    pay_state  = random.choices(
        ["not_paid", "partial", "paid"], weights=[0.50, 0.30, 0.20]
    )[0]
    if pay_state == "paid":
        residual = 0.0
    elif pay_state == "partial":
        residual = round(total * random.uniform(0.20, 0.70), 2)
    else:
        residual = total

    # sign: out_invoice positive, in_invoice negative
    sign = 1.0 if is_out else -1.0

    # credit note ref (~5 %)
    rev_entry = maybe(random.randint(max(1, move_id - 50), move_id - 1), 0.95) if move_id > move_id_start else None

    seq_pfx   = f"INV/{date.year}/" if is_out else f"BILL/{date.year}/"
    move_name = f"{seq_pfx}{move_id:05d}"

    move_rows.append(Row(
        account_move_id                = move_id,
        journal_id                     = jid,
        journal_name                   = jname,
        company_id                     = COMPANY_ID,
        company_name                   = COMPANY,
        partner_id                     = pid,
        partner_name                   = PARTNERS[pid],
        partner_shipping_id            = maybe(pid, 0.5),
        partner_shipping_name          = PARTNERS[pid] if random.random() > 0.5 else None,
        currency_id                    = cur_id,
        currency_name                  = cur_name,
        invoice_payment_term_id        = pt_id,
        invoice_payment_term_name      = PAYMENT_TERMS[pt_id],
        fiscal_position_id             = None,
        fiscal_position_name           = None,
        invoice_user_id                = user_id,
        invoice_user_name              = f"User_{user_id}",
        reversed_entry_id              = rev_entry,
        campaign_id                    = maybe(random.randint(1, 4), 0.7),
        campaign_name                  = maybe(f"Campaign_{random.randint(1,4)}", 0.7),
        team_id                        = maybe(random.randint(1, 3), 0.5),
        team_name                      = maybe(f"Team_{random.randint(1,3)}", 0.5),
        sequence_prefix                = seq_pfx,
        name                           = move_name,
        ref                            = maybe(f"REF-{random.randint(1000,9999)}", 0.5),
        state                          = "posted",
        move_type                      = move_type,
        auto_post                      = "no",
        payment_reference              = maybe(f"PAY-{random.randint(10000,99999)}", 0.4),
        payment_state                  = pay_state,
        invoice_origin                 = maybe(f"SO-{random.randint(1000,9999)}", 0.5),
        invoice_partner_display_name   = PARTNERS[pid],
        narration                      = maybe(f"Note {move_id}", 0.7),
        invoice_currency_rate          = cur_rate,
        amount_untaxed                 = base,
        amount_tax                     = tax_amount,
        amount_total                   = total,
        amount_residual                = residual,
        amount_untaxed_signed          = base * sign,
        amount_tax_signed              = tax_amount * sign,
        amount_total_signed            = total * sign,
        amount_residual_signed         = residual * sign,
        always_tax_exigible            = False,
        checked                        = False,
        posted_before                  = True,
        is_move_sent                   = pay_state != "not_paid",
        date                           = date,
        invoice_date                   = date,
        invoice_date_due               = date_due,
        delivery_date                  = del_date,
        auto_post_until                = None,
        created_at                     = date,
        updated_at                     = date,
        dwh_loaded_at                  = now,
        dwh_source_db                  = SOURCE_DB,
    ))

    # ── MOVE LINES (3 per move, always balanced) ─────────────
    prod_id  = random.randint(1, 100)
    uom_id   = random.choice(list(UOMS.keys()))
    qty      = float(random.randint(1, 20))
    price_u  = round(base / qty, 4)
    discount = random.choice([0.0] * 7 + [5.0, 10.0, 15.0])

    if is_out:
        line_defs = [
            # (account_type,    debit,  credit, prod,    is_tax, disp_type)
            ("asset_receivable", total,    0.0,   None,    False, "payment_term"),
            ("income",           0.0,     base,   prod_id, False, "product"),
            ("liability_current",0.0,     tax_amount, None, True, "tax"),
        ]
    else:
        line_defs = [
            ("expense",          base,     0.0,   prod_id, False, "product"),
            ("asset_current",    tax_amount, 0.0, None,    True,  "tax"),
            ("liability_payable",0.0,      total, None,    False, "payment_term"),
        ]

    for seq_n, (atype, debit, credit, lprod, is_tax, disp) in enumerate(line_defs, 1):
        balance = round(debit - credit, 2)
        tax_lid = 1 if is_tax else None
        tgrp_id = 1 if is_tax else None

        ml_rows.append(Row(
            account_move_line_id    = line_id,
            move_id                 = move_id,
            account_id              = pick_acc(atype),
            account_name            = NAMES_MAP[atype][0],
            journal_id              = jid,
            journal_name            = jname,
            company_id              = COMPANY_ID,
            company_name            = COMPANY,
            currency_id             = cur_id,
            currency_name           = cur_name,
            partner_id              = pid,
            partner_name            = PARTNERS[pid],
            product_id              = lprod,
            product_name            = PRODUCTS[lprod] if lprod else None,
            product_uom_id          = uom_id if lprod else None,
            product_uom_name        = UOMS[uom_id] if lprod else None,
            tax_line_id             = tax_lid,
            tax_line_name           = TAXES[tax_lid][0] if tax_lid else None,
            tax_group_id            = tgrp_id,
            tax_group_name          = f"Tax Group {tgrp_id}" if tgrp_id else None,
            payment_id              = None,
            sequence                = seq_n,
            move_name               = move_name,
            parent_state            = "posted",
            ref                     = None,
            name                    = disp,
            display_type            = disp,
            analytic_distribution   = None,
            debit                   = round(debit, 2),
            credit                  = round(credit, 2),
            balance                 = balance,
            amount_currency         = round(balance * cur_rate, 2),
            tax_base_amount         = base if is_tax else 0.0,
            amount_residual         = residual if disp == "payment_term" else 0.0,
            quantity                = qty if lprod else 0.0,
            price_unit              = price_u if lprod else 0.0,
            price_subtotal          = round(base, 2) if lprod else 0.0,
            price_total             = round(total, 2) if lprod else 0.0,
            discount                = discount if lprod else 0.0,
            is_storno               = False,
            reconciled              = pay_state == "paid",
            is_downpayment          = False,
            date                    = date,
            invoice_date            = date,
            date_maturity           = date_due if disp == "payment_term" else None,
            created_at              = date,
            updated_at              = date,
            dwh_loaded_at           = now,
            dwh_source_db           = SOURCE_DB,
        ))

        # Analytic line — only for product lines
        if lprod and ana_pool:
            ana_acc = random.choice(ana_pool)
            cat     = "invoice" if is_out else "expense"
            anal_rows.append(Row(
                analytic_line_id     = anal_id,
                account_id           = ana_acc,
                account_name         = f"ANA-{ana_acc:05d}",
                product_id           = lprod,
                product_name         = PRODUCTS[lprod],
                product_uom_id       = uom_id,
                product_uom_name     = UOMS[uom_id],
                partner_id           = pid,
                partner_name         = PARTNERS[pid],
                company_id           = COMPANY_ID,
                company_name         = COMPANY,
                currency_id          = cur_id,
                currency_name        = cur_name,
                journal_id           = jid,
                journal_name         = jname,
                general_account_id   = pick_acc(atype),
                general_account_name = NAMES_MAP[atype][0],
                name                 = move_name,
                category             = cat,
                code                 = f"ANA-{anal_id:06d}",
                ref                  = maybe(f"REF-{random.randint(1000,9999)}", 0.5),
                amount               = round(base, 2) * sign,
                unit_amount          = qty,
                date                 = date,
                created_at           = date,
                updated_at           = date,
                dwh_loaded_at        = now,
                dwh_source_db        = SOURCE_DB,
            ))
            anal_id += 1

        line_id += 1

    # ── PAYMENT  (only for paid / partial) ───────────────────
    if pay_state in ("paid", "partial"):
        pay_amount = total if pay_state == "paid" else (total - residual)
        p_type     = "inbound" if is_out else "outbound"
        p_partner  = "customer" if is_out else "supplier"
        pm_id      = random.choice(PM_POOL)
        rec        = pay_state == "paid"
        pay_rows.append(Row(
            account_payment_id              = pay_id,
            move_id                         = move_id,
            move_name                       = move_name,
            journal_id                      = random.choice(BANK_JIDS),
            journal_name                    = JOURNALS[random.choice(BANK_JIDS)][1],
            company_id                      = COMPANY_ID,
            company_name                    = COMPANY,
            partner_id                      = pid,
            partner_name                    = PARTNERS[pid],
            currency_id                     = cur_id,
            currency_name                   = cur_name,
            source_payment_id               = None,
            payment_method_id               = pm_id,
            payment_method_name             = PAYMENT_METHODS[pm_id],
            name                            = f"PAY/{date.year}/{pay_id:05d}",
            state                           = "posted",
            payment_type                    = p_type,
            partner_type                    = p_partner,
            memo                            = maybe(f"Payment for {move_name}", 0.4),
            payment_reference               = maybe(f"REF-{random.randint(10000,99999)}", 0.5),
            amount                          = round(pay_amount, 2),
            amount_company_currency_signed  = round(pay_amount * sign * cur_rate, 2),
            is_reconciled                   = rec,
            is_matched                      = rec,
            is_sent                         = True,
            date                            = date + timedelta(days=random.randint(0, 5)),
            created_at                      = date,
            updated_at                      = date,
            dwh_loaded_at                   = now,
            dwh_source_db                   = SOURCE_DB,
        ))
        pay_id += 1

    move_id += 1


# ═══════════════════════════════════════════════════════════════
# SCHEMAS
# ═══════════════════════════════════════════════════════════════

move_schema = StructType([
    StructField("account_move_id",               IntegerType(),   True),
    StructField("journal_id",                    IntegerType(),   True),
    StructField("journal_name",                  StringType(),    True),
    StructField("company_id",                    IntegerType(),   True),
    StructField("company_name",                  StringType(),    True),
    StructField("partner_id",                    IntegerType(),   True),
    StructField("partner_name",                  StringType(),    True),
    StructField("partner_shipping_id",           IntegerType(),   True),
    StructField("partner_shipping_name",         StringType(),    True),
    StructField("currency_id",                   IntegerType(),   True),
    StructField("currency_name",                 StringType(),    True),
    StructField("invoice_payment_term_id",       IntegerType(),   True),
    StructField("invoice_payment_term_name",     StringType(),    True),
    StructField("fiscal_position_id",            IntegerType(),   True),
    StructField("fiscal_position_name",          StringType(),    True),
    StructField("invoice_user_id",               IntegerType(),   True),
    StructField("invoice_user_name",             StringType(),    True),
    StructField("reversed_entry_id",             IntegerType(),   True),
    StructField("campaign_id",                   IntegerType(),   True),
    StructField("campaign_name",                 StringType(),    True),
    StructField("team_id",                       IntegerType(),   True),
    StructField("team_name",                     StringType(),    True),
    StructField("sequence_prefix",               StringType(),    True),
    StructField("name",                          StringType(),    True),
    StructField("ref",                           StringType(),    True),
    StructField("state",                         StringType(),    True),
    StructField("move_type",                     StringType(),    True),
    StructField("auto_post",                     StringType(),    True),
    StructField("payment_reference",             StringType(),    True),
    StructField("payment_state",                 StringType(),    True),
    StructField("invoice_origin",                StringType(),    True),
    StructField("invoice_partner_display_name",  StringType(),    True),
    StructField("narration",                     StringType(),    True),
    StructField("invoice_currency_rate",         DoubleType(),    True),
    StructField("amount_untaxed",                DoubleType(),    True),
    StructField("amount_tax",                    DoubleType(),    True),
    StructField("amount_total",                  DoubleType(),    True),
    StructField("amount_residual",               DoubleType(),    True),
    StructField("amount_untaxed_signed",         DoubleType(),    True),
    StructField("amount_tax_signed",             DoubleType(),    True),
    StructField("amount_total_signed",           DoubleType(),    True),
    StructField("amount_residual_signed",        DoubleType(),    True),
    StructField("always_tax_exigible",           BooleanType(),   True),
    StructField("checked",                       BooleanType(),   True),
    StructField("posted_before",                 BooleanType(),   True),
    StructField("is_move_sent",                  BooleanType(),   True),
    StructField("date",                          TimestampType(), True),
    StructField("invoice_date",                  TimestampType(), True),
    StructField("invoice_date_due",              TimestampType(), True),
    StructField("delivery_date",                 TimestampType(), True),
    StructField("auto_post_until",               TimestampType(), True),
    StructField("created_at",                    TimestampType(), True),
    StructField("updated_at",                    TimestampType(), True),
    StructField("dwh_loaded_at",                 TimestampType(), True),
    StructField("dwh_source_db",                 StringType(),    True),
])

ml_schema = StructType([
    StructField("account_move_line_id",  IntegerType(),   True),
    StructField("move_id",               IntegerType(),   True),
    StructField("account_id",            IntegerType(),   True),
    StructField("account_name",          StringType(),    True),
    StructField("journal_id",            IntegerType(),   True),
    StructField("journal_name",          StringType(),    True),
    StructField("company_id",            IntegerType(),   True),
    StructField("company_name",          StringType(),    True),
    StructField("currency_id",           IntegerType(),   True),
    StructField("currency_name",         StringType(),    True),
    StructField("partner_id",            IntegerType(),   True),
    StructField("partner_name",          StringType(),    True),
    StructField("product_id",            IntegerType(),   True),
    StructField("product_name",          StringType(),    True),
    StructField("product_uom_id",        IntegerType(),   True),
    StructField("product_uom_name",      StringType(),    True),
    StructField("tax_line_id",           IntegerType(),   True),
    StructField("tax_line_name",         StringType(),    True),
    StructField("tax_group_id",          IntegerType(),   True),
    StructField("tax_group_name",        StringType(),    True),
    StructField("payment_id",            IntegerType(),   True),
    StructField("sequence",              IntegerType(),   True),
    StructField("move_name",             StringType(),    True),
    StructField("parent_state",          StringType(),    True),
    StructField("ref",                   StringType(),    True),
    StructField("name",                  StringType(),    True),
    StructField("display_type",          StringType(),    True),
    StructField("analytic_distribution", StringType(),    True),
    StructField("debit",                 DoubleType(),    True),
    StructField("credit",                DoubleType(),    True),
    StructField("balance",               DoubleType(),    True),
    StructField("amount_currency",       DoubleType(),    True),
    StructField("tax_base_amount",       DoubleType(),    True),
    StructField("amount_residual",       DoubleType(),    True),
    StructField("quantity",              DoubleType(),    True),
    StructField("price_unit",            DoubleType(),    True),
    StructField("price_subtotal",        DoubleType(),    True),
    StructField("price_total",           DoubleType(),    True),
    StructField("discount",              DoubleType(),    True),
    StructField("is_storno",             BooleanType(),   True),
    StructField("reconciled",            BooleanType(),   True),
    StructField("is_downpayment",        BooleanType(),   True),
    StructField("date",                  TimestampType(), True),
    StructField("invoice_date",          TimestampType(), True),
    StructField("date_maturity",         TimestampType(), True),
    StructField("created_at",            TimestampType(), True),
    StructField("updated_at",            TimestampType(), True),
    StructField("dwh_loaded_at",         TimestampType(), True),
    StructField("dwh_source_db",         StringType(),    True),
])

pay_schema = StructType([
    StructField("account_payment_id",             IntegerType(),   True),
    StructField("move_id",                        IntegerType(),   True),
    StructField("move_name",                      StringType(),    True),
    StructField("journal_id",                     IntegerType(),   True),
    StructField("journal_name",                   StringType(),    True),
    StructField("company_id",                     IntegerType(),   True),
    StructField("company_name",                   StringType(),    True),
    StructField("partner_id",                     IntegerType(),   True),
    StructField("partner_name",                   StringType(),    True),
    StructField("currency_id",                    IntegerType(),   True),
    StructField("currency_name",                  StringType(),    True),
    StructField("source_payment_id",              IntegerType(),   True),
    StructField("payment_method_id",              IntegerType(),   True),
    StructField("payment_method_name",            StringType(),    True),
    StructField("name",                           StringType(),    True),
    StructField("state",                          StringType(),    True),
    StructField("payment_type",                   StringType(),    True),
    StructField("partner_type",                   StringType(),    True),
    StructField("memo",                           StringType(),    True),
    StructField("payment_reference",              StringType(),    True),
    StructField("amount",                         DoubleType(),    True),
    StructField("amount_company_currency_signed", DoubleType(),    True),
    StructField("is_reconciled",                  BooleanType(),   True),
    StructField("is_matched",                     BooleanType(),   True),
    StructField("is_sent",                        BooleanType(),   True),
    StructField("date",                           TimestampType(), True),
    StructField("created_at",                     TimestampType(), True),
    StructField("updated_at",                     TimestampType(), True),
    StructField("dwh_loaded_at",                  TimestampType(), True),
    StructField("dwh_source_db",                  StringType(),    True),
])

anal_schema = StructType([
    StructField("analytic_line_id",      IntegerType(),   True),
    StructField("account_id",            IntegerType(),   True),
    StructField("account_name",          StringType(),    True),
    StructField("product_id",            IntegerType(),   True),
    StructField("product_name",          StringType(),    True),
    StructField("product_uom_id",        IntegerType(),   True),
    StructField("product_uom_name",      StringType(),    True),
    StructField("partner_id",            IntegerType(),   True),
    StructField("partner_name",          StringType(),    True),
    StructField("company_id",            IntegerType(),   True),
    StructField("company_name",          StringType(),    True),
    StructField("currency_id",           IntegerType(),   True),
    StructField("currency_name",         StringType(),    True),
    StructField("journal_id",            IntegerType(),   True),
    StructField("journal_name",          StringType(),    True),
    StructField("general_account_id",    IntegerType(),   True),
    StructField("general_account_name",  StringType(),    True),
    StructField("name",                  StringType(),    True),
    StructField("category",              StringType(),    True),
    StructField("code",                  StringType(),    True),
    StructField("ref",                   StringType(),    True),
    StructField("amount",                DoubleType(),    True),
    StructField("unit_amount",           DoubleType(),    True),
    StructField("date",                  TimestampType(), True),
    StructField("created_at",            TimestampType(), True),
    StructField("updated_at",            TimestampType(), True),
    StructField("dwh_loaded_at",         TimestampType(), True),
    StructField("dwh_source_db",         StringType(),    True),
])


# ═══════════════════════════════════════════════════════════════
# WRITE ALL TO DELTA
# ═══════════════════════════════════════════════════════════════
print("\n── Writing transactional tables ──")
append_delta(spark.createDataFrame(acc_rows,  schema=acc_schema),  "account_account")
append_delta(spark.createDataFrame(ana_rows,  schema=ana_schema),  "account_analytic_account")
append_delta(spark.createDataFrame(bs_rows,   schema=bs_schema),   "account_bank_statement")
append_delta(spark.createDataFrame(bsl_rows,  schema=bsl_schema),  "account_bank_statement_line")
append_delta(spark.createDataFrame(move_rows, schema=move_schema), "account_move")
append_delta(spark.createDataFrame(ml_rows,   schema=ml_schema),   "account_move_line")
append_delta(spark.createDataFrame(pay_rows,  schema=pay_schema),  "account_payment")
append_delta(spark.createDataFrame(anal_rows, schema=anal_schema), "account_analytic_line")

print("\n✅  DONE — All 11 invoice Delta tables written to smart_odoo.silver")